# Detection Module

    The main goal of the detection module is to use the gazetteers out of the ontologies used to enrich PropaPhen into PropaPhen+ to discover relationships between network nodes/systems and the gufo:Entities by text.

In [1]:
%load_ext autoreload
%autoreload 2

## Libraries

### Installing

In [2]:
#!pip install pandas
#!pip install tqdm
#!pip install nltk
#!pip install gatenlp
#!pip install py4j
#!pip install pyodide
#!pip install ipywidgets
#!pip install neo4j

### Standard

In [3]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import nltk
import glob

In [4]:
from gatenlp import Document
from gatenlp.gateworker import GateWorker

### Custom libraries

In [5]:
import sys
sys.path.append('lib/')

In [6]:
from detection.relationshipextraction import RelationshipDiscovery, GateExtractor, CleanDicts, rmToRelationCSV
from detection.schema import Term, Concept, df_to_concepts, cleaningPlaceStr, conceptsToGazetteer
from detection.worldumls import umlsConceptCleanner, isEnglish, worldConceptCleanner
from detection.worldumls import ClearnWorldKGGazetteer
#import detection.observationclustering

## Globals

In [7]:
path_to_covid_journals = "data/textual/covid/newspaper/"
path_to_kb_gazetteer = '../data/gazetteers/kbgazetteer.csv'
path_to_netwoork_gazetteer = '../data/gazetteers/world_gazetteer_en.csv'
path_to_lsts = "data/lst/"
path_to_relation_folder = "../data/neo4j/"
path_to_observations = "../data/observations/"
path_to_covid_journalobservationcsv = "../data/neo4j/covid_observations_journal.csv"
path_to_covid_medicalobservationcsv = "../data/neo4j/covid_observations_medical.csv"
path_to_covid_socialobservationcsv = "../data/neo4j/covid_observations_social.csv"
path_to_monkeypox_journalobservationcsv = "../data/neo4j/monkeypox_observations_journal.csv"
path_to_monkeypox_medicalobservationcsv = "../data/neo4j/monkeypox_observations_medical.csv"
path_to_monkeypox_socialobservationcsv = "../data/neo4j/monkeypox_observations_social.csv"

## Relationship Discovery

### KB Gazetteers

In [8]:
kb_concept_list = []
network_concept_list = []

In [9]:
df_kb = pd.read_csv(path_to_kb_gazetteer)

In [10]:
df_kb.head()

,Unnamed: 0,ID,Name
0,0,C0026106,Mild mental retardation
1,1,C0026351,Moderate mental retardation
2,2,C0036857,Severe mental retardation
3,3,C0020796,Profound mental retardation
4,4,C0025362,Unspecified mental retardation


In [11]:
kb_concept_list = df_to_concepts(df_kb)

Finding Terms


12620098it [13:10, 15962.96it/s]


Creating Term list


In [12]:
for i in tqdm(range(len(kb_concept_list))):
    kb_concept_list[i] = umlsConceptCleanner(kb_concept_list[i])
    kb_concept_list[i] = umlsConceptCleanner(kb_concept_list[i])

100%|█████████████████████████████| 7892473/7892473 [00:17<00:00, 449268.59it/s]


In [13]:
umlsdict = conceptsToGazetteer(kb_concept_list,path_to_lsts+"umls.lst",cleaningPlaceStr)

100%|██████████████████████████████| 7892473/7892473 [02:25<00:00, 54088.17it/s]


### Place Gazetteers

In [14]:
df_network = pd.read_csv(path_to_netwoork_gazetteer)

In [15]:
washingtonRemoveDoubles = ('wkg:158368533', "Washington")
bradFord = ("wkg:26701367","Bradford")
def removeDoublesInNet(df_network,tupleList):
    list_id_to_remove = []
    for index, row in df_network.iterrows():
        for tupleRemoveDoubles in tupleList:
            if tupleRemoveDoubles[1] in row['Name'] and row['ID']!= tupleRemoveDoubles[0]:
                list_id_to_remove.append(row['ID'])

    df_network = df_network.drop(df_network[df_network.ID.isin(list_id_to_remove)].index.tolist())
    return df_network

In [16]:
df_network = removeDoublesInNet(df_network, [washingtonRemoveDoubles,bradFord])

In [17]:
clear_net_list = ['"Nga"', '"Centre"', '"Kou"', '"San"','"Real"',
                 '"Vincent"', '"Lille"','"North"', '"Barr"', '"North"'
                 ,'"South"','"West"','"East"','"Brito"', '"Utrecht"', '"Bush"',
                 '"Bush"', '"Republic"','"Union"', '"Time"',
                 '"Institute"','"Carbon"','"Center"','"Delhi"','"Mendenhall"']

In [18]:
df_network = ClearnWorldKGGazetteer(df_network,clear_net_list)

In [19]:
df_network.head()

,Unnamed: 0,ID,Name
0,0,wkg:10,"""Mamassita"""
1,1,wkg:10,"""Mamacita"""
2,2,wkg:1000709658,"""Boulzazen"""
3,3,wkg:1000709658,"""Boulzazen"""
4,4,wkg:1000709660,"""Tizi El Oued"""


In [20]:
network_concept_list = df_to_concepts(df_network)

Finding Terms


1692247it [01:56, 14498.51it/s]


Creating Term list


In [21]:
# Normal
print("Usual name")
normalplacesdict = conceptsToGazetteer(network_concept_list,path_to_lsts+"places.lst",cleaningPlaceStr)

Usual name


100%|███████████████████████████████| 948962/948962 [00:02<00:00, 339444.61it/s]


### GATE

In [22]:
gs = GateWorker(start=False, auth_token="1234")

2024-10-24 11:21:55,713|ERROR|py4j.java_gateway|An error occurred while trying to connect to the Java server (127.0.0.1:25333)
Traceback (most recent call last):
  File "/data/dataRapide/gabriel/git/DDPF/Detection/dtvenv/lib/python3.8/site-packages/py4j/java_gateway.py", line 982, in _get_connection
    connection = self.deque.pop()
IndexError: pop from an empty deque

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/data/dataRapide/gabriel/git/DDPF/Detection/dtvenv/lib/python3.8/site-packages/py4j/java_gateway.py", line 1132, in start
    self.socket.connect((self.address, self.port))
ConnectionRefusedError: [Errno 111] Connection refused


Py4JNetworkError: An error occurred while trying to connect to the Java server (127.0.0.1:25333)

In [ ]:
from nltk.corpus import stopwords
import string

def cleanKeys(dictionary, clean_list):
    for c in clean_list:
        if c in dictionary:
            del dictionary[c]
    return dictionary

def CleanDicts(netdict,kbdict):
    nltk.download('stopwords')
    stopwords_list = stopwords.words('english')
    punctuation = [i for i in string.punctuation  ]
    stopwords_list_maj = [s.title() for s in stopwords_list]
    months = ["January", "February", "March", "April", "May",
              "June", "July", "August", "September", "October", "November", "December"]
    months_lower = [m.lower() for m in months]
    clean_list = stopwords_list + punctuation + list(
        string.ascii_lowercase) + list(
        string.ascii_uppercase) + stopwords_list_maj + months + months_lower
    netdict = cleanKeys(netdict,clean_list) 
    kbdict = cleanKeys(kbdict,clean_list+list(netdict.keys()))
    return netdict, kbdict

In [ ]:
normalplacesdict, umlsdict = CleanDicts(normalplacesdict, umlsdict)

In [ ]:
gateExtractor = GateExtractor(umlsdict,normalplacesdict)

In [ ]:
# Annie
gs.worker.loadMavenPlugin("uk.ac.gate.plugins", "annie", "8.6")
# now load the prepared ANNIE pipeline from the plugin
pipeline = gs.worker.loadPipelineFromPlugin("uk.ac.gate.plugins","annie", "/resources/ANNIE_with_defaults.gapp")
pipeline.getName()

In [ ]:
gateExtractor.extra_pr['annie'] = pipeline

In [ ]:
from langchain_community.llms import Ollama
llm = Ollama(model="llama3")
def llamaCheck(kb, net,paragraph,llm=llm):
    improvedParagraph = paragraph.replace(kb,"XXX")
    improvedParagraph = improvedParagraph.replace(net,"YYY")
    improvedParagraph = improvedParagraph.replace('"',"'")
    prompt = '/clear .In the phrase "'+improvedParagraph+'", only considering the phrase, is "XXX" positively related to "YYY"? (Answer with yes or no only)'
    result = llm.invoke(prompt)
    return ('yes' in result.split('\n')[0].lower())

In [ ]:
from detection.relationshipextraction import RelationMatrix

class RMGenerator():
    
    def __init__(self, corpus,gateExtractor, gs):
        self.corpus = corpus
        self.gateExtractor = gateExtractor
        self.gs = gs
    
    def directTermMatching(self, matrix_id):
        rm = RelationMatrix(matrix_id)
        # Per document
        for doc in tqdm(self.corpus):
            pdoc = self.gs.gdoc2pdoc(doc)
            pdoc = self.gateExtractor.tokenizer(pdoc)
            pdoc = self.gateExtractor.tok_gaz(pdoc)
            # Making the rm links
            for kb_annotation in pdoc.annset().with_type("kb"):
                for network_annoation in pdoc.annset().with_type("network"):
                    rm.increaseBy(self.gateExtractor.dict_kb[kb_annotation.features['key']], 
                                self.gateExtractor.dict_network[network_annoation.features['key']],1)
            self.gs.del_resource(doc)
        return rm
    
    def paragraphTermMatching(self, matrix_id):
        assert 'annie' in self.gateExtractor.extra_pr.keys()
        rm = RelationMatrix(matrix_id)
        # Per document
        for doc in tqdm(self.corpus):
            # Run annie
            if len(self.gs.gdoc2pdoc(doc).text) <= 0:
                self.gs.del_resource(doc)
                continue
            self.gs.worker.run4Document(self.gateExtractor.extra_pr['annie'], doc)
            pdoc = self.gs.gdoc2pdoc(doc)            
            # Get network and kb
            pdoc = self.gateExtractor.tok_gaz(pdoc)
            # Get paragraph
            praragraphann = pdoc.annset('Original markups').with_type("paragraph")
            # For each paragraph
            for ann in praragraphann:
                # Making the rm links
                for kb_annotation in pdoc.annset().within(ann).with_type('kb'):
                    for network_annoation in pdoc.annset().within(ann).with_type('network'):
                        rm.increaseBy(self.gateExtractor.dict_kb[kb_annotation.features['key']], 
                                self.gateExtractor.dict_network[network_annoation.features['key']],1)
            self.gs.del_resource(doc)
        return rm
    
    def paragraphTermMatchingTransitivity(self, matrix_id):
        assert 'annie' in self.gateExtractor.extra_pr.keys()
        rm = RelationMatrix(matrix_id)
        # Per document
        for doc in tqdm(self.corpus):
            # Run annie
            if len(self.gs.gdoc2pdoc(doc).text) <= 0:
                self.gs.del_resource(doc)
                continue
            self.gs.worker.run4Document(self.gateExtractor.extra_pr['annie'], doc)
            pdoc = self.gs.gdoc2pdoc(doc)            
            # Get network and kb
            pdoc = self.gateExtractor.tok_gaz(pdoc)
            # Get paragraph
            paragraphann = pdoc.annset('Original markups').with_type("paragraph")
            dictKbToKb = {}
            # For each paragraph
            for ann in paragraphann:
                # Making Refs between KB entities
                for kb_annotation1 in pdoc.annset().within(ann).with_type('kb'):
                    for kb_annotation2 in pdoc.annset().within(ann).with_type('kb'):
                        # if same annotation continue
                        if kb_annotation1 == kb_annotation2:
                            continue
                        # If empty list create list
                        if kb_annotation1.features['key'] not in dictKbToKb:
                                dictKbToKb[kb_annotation1.features['key']] = []
                        # Add key to list
                        dictKbToKb[kb_annotation1.features['key']] = dictKbToKb[
                            kb_annotation1.features['key']] +  [kb_annotation2.features['key']]
            for ann in paragraphann:
                # Making the rm links
                for kb_annotation in pdoc.annset().within(ann).with_type('kb'):
                    for network_annoation in pdoc.annset().within(ann).with_type('network'):
                        rm.increaseBy(self.gateExtractor.dict_kb[kb_annotation.features['key']], 
                                self.gateExtractor.dict_network[network_annoation.features['key']],1)
            # Adding transitivity relations
            for ann in paragraphann:
                # Making the rm links
                for network_annoation in pdoc.annset().within(ann).with_type('network'):
                    for kb_annotation in pdoc.annset().within(ann).with_type('kb'):
                        if kb_annotation.features['key'] not in dictKbToKb:
                            continue
                        for transitivityKey in dictKbToKb[kb_annotation.features['key']]:
                            if rm.getValue(self.gateExtractor.dict_kb[transitivityKey], 
                                self.gateExtractor.dict_network[network_annoation.features['key']]) is None:
                                # If link does not exists, then create one
                                rm.increaseBy(self.gateExtractor.dict_kb[transitivityKey], 
                                self.gateExtractor.dict_network[network_annoation.features['key']],1)
                            
            self.gs.del_resource(doc)
        return rm
    
    def sentenceTermMatching(self, matrix_id):
        assert 'annie' in self.gateExtractor.extra_pr.keys()
        rm = RelationMatrix(matrix_id)
        # Per document
        for doc in tqdm(self.corpus):
            # Run annie
            if len(self.gs.gdoc2pdoc(doc).text) <= 0:
                self.gs.del_resource(doc)
                continue
            self.gs.worker.run4Document(self.gateExtractor.extra_pr['annie'], doc)
            pdoc = self.gs.gdoc2pdoc(doc)            
            # Get network and kb
            pdoc = self.gateExtractor.tok_gaz(pdoc)
            # Get paragraph
            sentenceann = pdoc.annset('').with_type("Sentence")
            # For each paragraph
            for ann in sentenceann:
                # Making the rm links
                for kb_annotation in pdoc.annset().within(ann).with_type('kb'):
                    for network_annoation in pdoc.annset().within(ann).with_type('network'):
                        rm.increaseBy(self.gateExtractor.dict_kb[kb_annotation.features['key']], 
                                self.gateExtractor.dict_network[network_annoation.features['key']],1)
            self.gs.del_resource(doc)
        return rm
    
    def sentenceTermMatchingTransitivity(self, matrix_id):
        assert 'annie' in self.gateExtractor.extra_pr.keys()
        rm = RelationMatrix(matrix_id)
        # Per document
        for doc in tqdm(self.corpus):
            # Run annie
            if len(self.gs.gdoc2pdoc(doc).text) <= 0:
                self.gs.del_resource(doc)
                continue
            self.gs.worker.run4Document(self.gateExtractor.extra_pr['annie'], doc)
            pdoc = self.gs.gdoc2pdoc(doc)            
            # Get network and kb
            pdoc = self.gateExtractor.tok_gaz(pdoc)
            # Get paragraph
            sentenceann = pdoc.annset('').with_type("Sentence")
            dictKbToKb = {}
            # For each paragraph
            for ann in sentenceann:
                # Making Refs between KB entities
                for kb_annotation1 in pdoc.annset().within(ann).with_type('kb'):
                    for kb_annotation2 in pdoc.annset().within(ann).with_type('kb'):
                        # if same annotation continue
                        if kb_annotation1 == kb_annotation2:
                            continue
                        # If empty list create list
                        if kb_annotation1.features['key'] not in dictKbToKb:
                                dictKbToKb[kb_annotation1.features['key']] = []
                        # Add key to list
                        dictKbToKb[kb_annotation1.features['key']] = dictKbToKb[
                            kb_annotation1.features['key']] +  [kb_annotation2.features['key']]
            # For each paragraph
            for ann in sentenceann:
                # Making the rm links
                for kb_annotation in pdoc.annset().within(ann).with_type('kb'):
                    for network_annoation in pdoc.annset().within(ann).with_type('network'):
                        rm.increaseBy(self.gateExtractor.dict_kb[kb_annotation.features['key']], 
                                self.gateExtractor.dict_network[network_annoation.features['key']],1)
            # Adding transitivity relations
            for ann in sentenceann:
                # Making the rm links
                for network_annoation in pdoc.annset().within(ann).with_type('network'):
                    for kb_annotation in pdoc.annset().within(ann).with_type('kb'):
                        if kb_annotation.features['key'] not in dictKbToKb:
                            continue
                        for transitivityKey in dictKbToKb[kb_annotation.features['key']]:
                            if rm.getValue(self.gateExtractor.dict_kb[transitivityKey], 
                                self.gateExtractor.dict_network[network_annoation.features['key']]) is None:
                                # If link does not exists, then create one
                                rm.increaseBy(self.gateExtractor.dict_kb[transitivityKey], 
                                self.gateExtractor.dict_network[network_annoation.features['key']],1)
            self.gs.del_resource(doc)
        return rm
    
    def llama3(self, matrix_id):
        assert 'annie' in self.gateExtractor.extra_pr.keys()
        rm = RelationMatrix(matrix_id)
        # Per document
        for doc in tqdm(self.corpus):
            # Run annie
            if len(self.gs.gdoc2pdoc(doc).text) <= 0:
                self.gs.del_resource(doc)
                continue
            self.gs.worker.run4Document(self.gateExtractor.extra_pr['annie'], doc)
            pdoc = self.gs.gdoc2pdoc(doc)            
            # Get network and kb
            pdoc = self.gateExtractor.tok_gaz(pdoc)
            # Get paragraph
            praragraphann = pdoc.annset('Original markups').with_type("paragraph")
            document_text = pdoc.text
            # For each paragraph
            for ann in praragraphann:
                # Making the rm links
                for kb_annotation in pdoc.annset().within(ann).with_type('kb'):
                    for network_annoation in pdoc.annset().within(ann).with_type('network'):
                        start = rmSentence.start
                        end = rmSentence.end
                        hasPresence = llamaCheck(
                        str(self.gateExtractor.dict_kb[kb_annotation.features['key']]),
                            str(self.gateExtractor.dict_network[network_annoation.features['key']]),
                            document_text[start:end]
                        )
                        if hasPresence:
                            rm.increaseBy(self.gateExtractor.dict_kb[kb_annotation.features['key']], 
                                    self.gateExtractor.dict_network[network_annoation.features['key']],1)
            self.gs.del_resource(doc)
        return rm

In [ ]:
class RelationshipDiscovery():
    
    def __init__(self,corpus, gateExtractor, gs, rmGen=None):
        self.corpus = corpus
        self.gateExtractor = gateExtractor
        if rmGen is not None:
            self.rmGen = rmGen
            assert self.rmGen.corpus == self.corpus
            assert self.rmGen.gateExtractor == self.gateExtractor
        else:
            self.rmGen = RMGenerator(self.corpus, self.gateExtractor, gs)

## Detection Benchmark

### Setup

Two current points of attention:
1. Don't forget to rename the corpus on gate to match the paths in this code!
2. For each corpus, open it, run the code related to it, then go to the next. The current getCorpus4Name from gate is not working!

#### COVID

In [ ]:
# Corpus Journal
corpusCOVIDJournal = gs.getCorpus4Name('COVID-Journal-11-19')

In [ ]:
# Corpus Medical
corpusCOVIDMedical = gs.getCorpus4Name('COVID-Medical-12-19')

In [ ]:
# Corpus Social
corpusCOVIDSocial = gs.getCorpus4Name('COVID-Social-02-20')

#### Monkeypox

In [ ]:
# Corpus Journal
corpusMonkeypoxJournal = gs.getCorpus4Name('Monkeypox-Journal-05-22')

In [ ]:
# Corpus Medical
corpusMonkeypoxMedical = gs.getCorpus4Name('Monkeypox-Medical-06-22')

In [ ]:
# Corpus Social
corpusMonkeypoxSocial = gs.getCorpus4Name('Monkeypox-Social-05-22')

In [ ]:
corpusDict = {
    ("COVID", "Journal", "11-19") : corpusCOVIDJournal,
    ("COVID", "Medical", "12-19") : corpusCOVIDMedical,
    ("COVID", "Social", "02-20") : corpusCOVIDSocial,
    ("Monkeypox", "Journal", "05-22") : corpusMonkeypoxJournal,
    ("Monkeypox", "Medical", "06-22") : corpusMonkeypoxMedical,
    ("Monkeypox", "Social", "05-22") : corpusMonkeypoxSocial
}

### Relationship Discovery - COVID

In [ ]:
class DetectionBenchmarkEntry():
    
    def __init__(self, phenomenon, date, source, method, value):
        self.phenomenon = phenomenon
        self.date = date
        self.source = source
        self.method = method
        self.value = value

class DetectionBenchmark():
    
    def __init__(self):
        self.entry_list = []
        
    def addEntry(self, benchEntry):
        self.entry_list.append(benchEntry)
        
    def removeEntry(self, benchEntry):
        if benchEntry in self.entry_list:
            self.entry_list.remove(benchEntry)
            
    def benchmarkToPandas(self):
        phenomenon_list = []
        date_list = []
        source_list = []
        method_list = []
        value_list = []
        for entry in self.entry_list:
            phenomenon_list.append(entry.phenomenon)
            date_list.append(entry.date)
            source_list.append(entry.source)
            method_list.append(entry.method)
            value_list.append(entry.value)
        return pd.Dataframe( {
            'phenomenon' : phenomenon_list,
            'date' : date_list,
            'source' : source_list,
            'method_list' : method_list,
            'value' : value_list
        })

In [ ]:
def AddMethodToBenchMark(rd, rmGen,
                         detectionBenchmark, phenomenon_type,
                         source_type,date,method_type,save_csv=True):
    df_rmGen = rmToRelationCSV(rmGen,
                            source_type+'_'+phenomenon_type+'_'+method_type,
                            1, 'hasPresence',cluster_date=date)
    if save_csv:
        df_rmGen.to_csv(path_to_relation_folder+rmGen.matrix_id+".csv", index=False)
    entry = DetectionBenchmarkEntry(phenomenon_type, date, source_type, method_type, 0)
    detectionBenchmark.addEntry(entry)

def CreateBenchMark(gs, corpusDict,gateExtractor,
                    path_to_relation_folder=path_to_relation_folder):
    detectionBenchmark = DetectionBenchmark()
    for tuplePhenomenon in corpusDict.keys():
        (phenomenon_type, source_type, date) = tuplePhenomenon
        corpus = corpusDict[tuplePhenomenon]
        rd = RelationshipDiscovery(corpus, gateExtractor,gs)
        # Document Matching
        method_type = "DocumentMatching"
        print((phenomenon_type+'-'+source_type+'-'+date+'-'+method_type))
        rmGen = rd.rmGen.directTermMatching(phenomenon_type+'-'+source_type+'-'+date+'-'+method_type)
        AddMethodToBenchMark(rd, rmGen, detectionBenchmark, phenomenon_type,source_type,date,method_type)
        # Paragraph Matching
        method_type = "ParagraphMatching"
        print((phenomenon_type+'-'+source_type+'-'+date+'-'+method_type))
        rmGen = rd.rmGen.paragraphTermMatching(phenomenon_type+'-'+source_type+'-'+date+'-'+method_type)
        AddMethodToBenchMark(rd, rmGen, detectionBenchmark, phenomenon_type,source_type,date,method_type)
        # Sentence Matching
        method_type = "SentenceMatching"
        print((phenomenon_type+'-'+source_type+'-'+date+'-'+method_type))
        rmGen = rd.rmGen.sentenceTermMatching(phenomenon_type+'-'+source_type+'-'+date+'-'+method_type)
        AddMethodToBenchMark(rd, rmGen, detectionBenchmark, phenomenon_type,source_type,date,method_type)
    return detectionBenchmark

### Relationship Discovery - Monkeypox

In [ ]:
detectionBenchmark = CreateBenchMark(gs, corpusDict,gateExtractor,path_to_relation_folder)

### Observation Mining

In [ ]:
from lib.kgce.schema.semantic.neo4jclasses import Neo4jRelation
from lib.kgce.neo4j.handler import Neo4jWrapper

In [ ]:
from neo4j import GraphDatabase
from tqdm import tqdm


class Neo4jWrapper:

    def __init__(self, uri, userName, password):
        self.uri = uri
        self.userName = userName
        self.password = password
        # Connect to the neo4j database server
        self.graphDB_Driver  = GraphDatabase.driver(uri, auth=(userName, password)) 
        
    def sendQuery(self, cql_commands):
        result = []
        done_queries = []
        with self.graphDB_Driver.session() as graphDB_Session:
            for cqlCreate in tqdm(cql_commands):
                try:
                    result += [graphDB_Session.run(cqlCreate).to_df()]
                    done_queries.append(cqlCreate)
                except Exception as e:
                    tqdm.write(str(e))
                    tqdm.write(cqlCreate)
                    result += [str(e)]
        return result
    
    def closeConnection(self):
        self.graphDB_Driver.close()

In [ ]:
neowrapper = Neo4jWrapper(uri="bolt://localhost:7687",userName="neo4j",password="test")

In [ ]:
def GetObservationFromSource(neowrapper,source, filterValue):
    strQuery = """MATCH (n:Country)<-[r:hasPresence]-(c) 
        WHERE toInteger(r.intensity) >= {0} AND r.source = "{1}"
        RETURN n.wkgs_nameEn as System_Name, n.id, c.name, c.id, r.intensity as intensity;""".format(
        filterValue, source)
    result = neowrapper.sendQuery([strQuery])
    df_result_journal = result[0].groupby(['System_Name','n.id'],as_index=False).agg(list)
    return df_result_journal

#### 1 Experiments on COVID on Journal

In [ ]:
# Evaluation Terms
covidEvTerms = {
    'Pneumonia' : ['pneumonia', 'respiratory outbreak',
                   'lung disease',
                   'respiratory tract illness',
                   'respiratory illness',
                   'respiratory infection',
                   'pneumonia-like disease',
                   'upper-respiratory illness',
                   'respiratory condition',
                   'lung infection',
                   'pneumonia-like cases',
                   'pneumonia-like illness',
                   'respiratory virus',
                   'lung virus',
                   'pneumonia-like virus',
                   'pneumonia-causing virus',
                   'pneumonia-like virus'
    ],
    'Mistery' : ['mistery', 'mmisterious',
                 'unidentified',
                 'undocumented',
                 'disease x',
                 'unknown',
                 'abnormal',
                 'unexplained'        
    ],
    'Technical' : ['2019-ncov', 'ncov',
                   '2019 novel coronavirus',
                   'n-cov2019', 'novel coronavirus 2019',
                   'ncov2019', 'cov2019'
    ],
    'Coronavirus' : ['coronavirus', 'betacoronavirus',
                     'coronovirus'
    ],
    'Family' : ['sars', 'severe acute respiratory syndrome',
               'sars coronavirus', 'sars-cov']
    
}

In [ ]:
def ObsScore(df_observation,evTerms):
    dict_result = {}
    # verify if any country has term
    for index, row in df_observation.iterrows():
        country_name = row['System_Name']
        dict_country = {}
        list_of_terms = row['c.name']
        for key in evTerms.keys():
            dict_country[key] = 0
            for obsterm in list_of_terms:
                vocab = evTerms[key]
                # For every term from vocab
                for vocabterm in vocab:
                    if vocabterm.lower() in obsterm.lower():
                        dict_country[key] = 1
        dict_result[country_name] = dict_country
    return dict_result

In [ ]:
from sklearn.metrics import f1_score, precision_score
import numpy as np
def obsMetricScore(obsScoreDict, category):
    y_pred = []
    y_true = []
    for key in obsScoreDict.keys():
        y_pred.append(obsScoreDict[key][category])
        if key == '"China"':
            y_true.append(1)
        else:
            y_true.append(0)
    class_counts = np.bincount(y_true)
    inverse_class_proportions = 1.0 / class_counts
    inverse_class_weights = inverse_class_proportions / sum(inverse_class_proportions)
    f1_class0 = f1_score(y_true, y_pred, pos_label=0)
    f1_class1 = f1_score(y_true, y_pred, pos_label=1)
    f1_inverse = f1_class0 * inverse_class_weights[0] + f1_class1 * inverse_class_weights[1]

    return f1_score(y_true, y_pred, average='weighted')

In [ ]:
# Retrieving all (source_type, phenomenon_type,method_type)-triples
strQuery = """MATCH (a)-[r:hasPresence]->(b) RETURN DISTINCT (r.source);"""
result = neowrapper.sendQuery([strQuery])

In [ ]:
sourceList = list(result[0]['(r.source)'])
print(sourceList)

In [ ]:
dict_observations =  {}
for source in sourceList:
    print("Mining " + source +" Observation")
    threshold = 5
    if "Journal_COVID" in source:
        threshold = 5
    elif "Medical_COVID" in source:
        threshold = 8
    elif "Social_COVID" in source:
        threshold = 10
    elif "Journal_Monkeypox" in source:
        threshold = 5
    elif "Medical_Monkeypox" in source:
        threshold = 3
    elif "Social_Monkeypox" in source:
        threshold = 10
    df_observation = GetObservationFromSource(neowrapper,source,threshold)
    dict_observations[source] = df_observation
    df_observation.to_csv(path_to_observations+"/"+source+"_Observation.csv")

In [ ]:
dict_table = {'Pneumonia' : [] , 'Mistery' : [], 'Technical' : [], 'Coronavirus' : [], 'Family' : []}
list_of_obs = [dict_observations['Journal_COVID_DocumentMatching'],
               dict_observations['Journal_COVID_ParagraphMatching'],
               dict_observations['Journal_COVID_SentenceMatching']]
for obsMatrix in list_of_obs:
    obsScoreDict = ObsScore(obsMatrix,covidEvTerms)
    for key in covidEvTerms.keys():
        dict_table[key] = dict_table[key] +  [obsMetricScore(obsScoreDict,key)]

In [ ]:
df_result = pd.DataFrame(dict_table,
                  index=pd.Index(['Document Matching', 'Paragraph Matchin', 'Sentence Matching']))
df_result